## Synergy of mergerd Kabuki 1 & 2 syndrome cohorts

In [2]:
from ppktstore.registry import configure_phenopacket_registry
import pandas as pd
from phenosign import PhenotypeDatasetBuilder, HPOCorrelationAnalyzer, SynergyAnalyzer

In [8]:
registry = configure_phenopacket_registry()
with registry.open_phenopacket_store() as ps:
    kmt2d = list(ps.iter_cohort_phenopackets("KMT2D"))
    kdm6a = list(ps.iter_cohort_phenopackets("KDM6A"))

phenopackets = kmt2d + kdm6a
print(f"KMT2D={len(kmt2d)}, KDM6A={len(kdm6a)}, combined={len(phenopackets)}")

KMT2D=64, KDM6A=81, combined=145


In [9]:
dataset = PhenotypeDatasetBuilder(phenopackets).build(build_gpsea_cohort=False)

# Inspect what's actually in the merged data before hardcoding IDs:
diseases_df, sex_df, genes_df, _ = dataset.describe_conditions()
print(diseases_df)  

                         label n_individuals
disease_id                                  
OMIM:300867  Kabuki syndrome 2    81 (55.9%)
OMIM:147920  Kabuki Syndrome 1    64 (44.1%)


In [11]:
analyzer = HPOCorrelationAnalyzer(dataset=dataset)

results = analyzer.compute_correlation_matrix()

results.results_table.head()

Calculating pairwise correlation: 100%|██████████| 877/877 [00:01<00:00, 570.43it/s]


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,correlation,p_value,adj_p_value,n(A:E/B:E),n(A:E/B:O),n(A:O/B:E),n(A:O/B:O),n_individuals,n_pmids,pmids
255,HP:0000750,Delayed speech and language development,HP:0001270,Motor delay,0.679803,0.000050,0.030268,5,2,2,56,65,4,31949313;32083401;33674768;35060672
368,HP:0001252,Hypotonia,HP:0011968,Feeding difficulties,0.487063,0.000196,0.058817,10,8,5,46,69,4,25896430;29482518;31846209;33674768
251,HP:0000750,Delayed speech and language development,HP:0001249,Intellectual disability,0.758288,0.000480,0.096038,3,2,0,46,51,3,24019847;32083401;33674768
562,HP:0002342,Moderate intellectual disability,HP:0010864,Severe intellectual disability,-0.467707,0.001807,0.157549,15,21,9,0,45,3,25896430;31727177;33674768
302,HP:0001212,Prominent fingertip pads,HP:0011968,Feeding difficulties,0.495416,0.001838,0.157549,7,6,3,32,48,4,29482518;31727177;31846209;33674768


In [12]:
results.save_correlation_results(
    output_file="../case_studies/KMT2D_KDM6A/correlation_results_KMT2D_KDM6A.csv"
)

In [13]:
results.plot_correlation_heatmap_with_significance(
    title_name="Cohort FBN1",
) 

In [14]:
results.save_correlation_heatmap()

### **Synergy Analysis Between HPO Terms and Target Variables**

In [15]:
diseases_df, sex_df, genes_df, variant_effects_df = dataset.describe_conditions()

diseases_df

,label,n_individuals
disease_id,,
OMIM:300867,Kabuki syndrome 2,81 (55.9%)
OMIM:147920,Kabuki Syndrome 1,64 (44.1%)


In [16]:
sex_df

,n_individuals
sex,
female,81 (55.9%)
male,64 (44.1%)
unknown,0 (0.0%)


In [17]:
genes_df

,n_individuals
gene_symbol,
KDM6A,81 (55.9%)
KMT2D,64 (44.1%)


In [18]:
variant_effects_df

In [19]:
synergy_analyzer = SynergyAnalyzer(dataset=dataset)

In [23]:
from phenosign import has_disease
condition = dataset.get_condition(
    has_disease("OMIM:300867"),
    name="Kabuki 2 (KDM6A) vs Kabuki 1 (KMT2D)",
)

synergy_analyzer = SynergyAnalyzer(dataset=dataset)
results = synergy_analyzer.compute_synergy_matrix(condition=condition)
results.results_table.head()
results.plot_synergy_heatmap()

Calculating pairwise synergy: 100%|██████████| 503/503 [00:11<00:00, 44.09it/s] 
